In [1]:
import os
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader


DATASETS_DIR = '/Users/keerthana/Keerthana/workspace/RS-hackathon/clinical-ai-assistanat/datasets'


def load_pdf_text(path):
    loader = PyPDFLoader(path)
    pdf_doc = loader.load()
    text = ""
    for doc in pdf_doc:
        text += doc.page_content
    return text


def load_csv_text(path):
    df = pd.read_csv(path)
    return df.to_string(index=False)


def load_all_disease_data():
    """Load all text data from PDFs + CSVs for each disease."""
    disease_data = {}

    for disease in os.listdir(DATASETS_DIR):
        disease_path = os.path.join(DATASETS_DIR, disease)
        if not os.path.isdir(disease_path):
            continue

        disease_texts = []
        for file in os.listdir(disease_path):
            file_path = os.path.join(disease_path, file)
            if file.endswith(".pdf"):
                disease_texts.append(load_pdf_text(file_path))
            elif file.endswith(".csv"):
                disease_texts.append(load_csv_text(file_path))

        disease_data[disease] = "\n\n".join(disease_texts)
    return disease_data

/Users/keerthana/Keerthana/workspace/RS-hackathon/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


def build_vector_store(disease_data):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    embeddings = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    
    vectordb = Chroma(
        embedding_function=embeddings,
        collection_name="disease_studies",
        persist_directory="./chroma/disease_studies"
    )
    
    for disease, data in disease_data.items():
        print(f"Processing disease: {disease}")
        # Split the text into chunks
        chunks = splitter.split_text(data)
        # Convert chunks to Document objects
        documents = [
            Document(
                page_content=chunk,
                metadata={"disease": disease}
            ) for chunk in chunks
        ]
        # Add documents in batches
        vectordb.add_documents(documents)

    vectordb.persist()
    return vectordb

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from typing import Dict, List


def format_docs(docs) -> str:
    """Format documents with their metadata into a string."""
    return "\n\n".join([f"{doc.metadata.get('disease', 'Unknown')}: {doc.page_content}" for doc in docs])


def retrival(vectordb, query: str) -> Dict:
    """
    Retrieve and answer questions using RAG (Retrieval Augmented Generation).
    
    Args:
        vectordb: The vector store containing the documents
        query: The question to answer
        
    Returns:
        Dict containing the answer and source documents
    """
    # Create retriever with MMR search for better diversity
    retriever = vectordb.as_retriever(
        search_type="mmr",  # Using MMR for better result diversity
        search_kwargs={"k": 5}
    )

    # Initialize the language model
    llm = ChatOpenAI(
        model_name="gpt-4",
        temperature=0
    )

    # Create the prompt template
    template = """You are a trusted clinical assistant.
    Use ONLY the provided context to answer the question.
    If you are unsure or information is missing, say:
    "I don't have enough data to answer that confidently."

    Context:
    {context}

    Question: {question}
    
    Provide a comprehensive answer based on the context above.
    
    Answer:"""
    
    prompt = PromptTemplate.from_template(template)

    # Create the RAG chain
    rag_chain = (
        {
            "context": lambda x: format_docs(retriever.get_relevant_documents(x)),
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    # Get answer and source documents
    answer = rag_chain.invoke(query)
    
    return answer

In [ ]:
diseases_data = load_all_disease_data()
print("Loaded disease data.")

vectordb = build_vector_store(diseases_data)


In [ ]:
query = "What are the best predictive algorithms for heart attack detection?"
answer = retrival(vectordb, query)
print("Answer:", answer)